# Phase 2: Stochastic Modelling & Derivatives Pricing  
## Notebook 02_13 — Multilevel Monte Carlo Pricing

### Research question

How can Multilevel Monte Carlo reduce the computational cost of pricing path-dependent derivatives compared with standard Monte Carlo?

This notebook extends:

```text
02_06_monte_carlo_gbm_and_fat_tails
02_07_exotic_options_monte_carlo
02_10_finite_difference_pde_pricer
```

The previous Monte Carlo notebooks priced derivatives by simulating many paths at one fixed resolution. This notebook introduces **Multilevel Monte Carlo** (MLMC), which combines many cheap coarse simulations with fewer expensive fine simulations.

It covers:

1. standard Monte Carlo pricing;
2. path discretisation bias;
3. MLMC telescoping identity;
4. coupled coarse/fine Brownian paths;
5. Asian option payoff under GBM;
6. level-difference variance decay;
7. cost-per-level diagnostics;
8. pilot estimation;
9. optimal MLMC sample allocation;
10. MLMC estimator construction;
11. comparison against standard Monte Carlo;
12. bias/variance/cost trade-off;
13. limitations and production extensions.

The key idea:

> MLMC uses cheap coarse paths to estimate most of the expectation and expensive fine paths only to estimate small correction terms.

## 1. Why standard Monte Carlo can be expensive

For a derivative payoff $P$, standard Monte Carlo estimates:

$$
\begin{aligned}
\mathbb E[P] &\approx \frac{1}{N}\sum_{i=1}^N P^{(i)}
\end{aligned}
$$
The standard error is:

$$
\frac{\sqrt{\operatorname{Var}(P)}}{\sqrt{N}}
$$
So to reduce error by a factor of 10, we need about 100 times more paths.

For path-dependent derivatives, each path also requires many time steps. If $M$ is the number of time steps and $N$ is the number of paths, the cost is approximately:

$$
\text{cost} = O(NM)
$$
This becomes expensive when pricing products such as:

- Asian options;
- barrier options;
- lookback options;
- discretely hedged portfolios;
- XVA exposures;
- stochastic-volatility products.

## 2. Path discretisation and levels

Suppose a fine-grid payoff approximation is:

$$
P_L
$$
where level $L$ uses:

$$
M_L = M_0 2^L
$$
time steps.

A standard Monte Carlo estimator at level $L$ estimates:

$$
\mathbb E[P_L]
$$
directly using many fine paths.

MLMC uses the telescoping identity:

$$
\begin{aligned}
\mathbb E[P_L] &= \mathbb E[P_0] \\
&\quad + \sum_{\ell=1}^{L} \mathbb E[P_\ell-P_{\ell-1}]
\end{aligned}
$$
Each difference:

$$
P_\ell-P_{\ell-1}
$$
is computed using **coupled paths** driven by the same Brownian motion.

If the fine and coarse paths are strongly coupled, the variance of the difference decays with level:

$$
\operatorname{Var}(P_\ell-P_{\ell-1}) \to 0
$$
Then high levels need fewer samples.

## 3. Asian option test case

We use an arithmetic Asian call under risk-neutral GBM.

The payoff is:

$$
\begin{aligned}
P &= e^{-rT} \Bigg( \frac{1}{M} \sum_{j=1}^{M}S_{t_j} \\
&\quad - K \Bigg)^+
\end{aligned}
$$
This is a good MLMC demonstration because:

1. the payoff is path-dependent;
2. standard Monte Carlo needs full paths;
3. coarse and fine path averages can be coupled naturally;
4. the exact arithmetic Asian price is not generally closed form;
5. a high-resolution Monte Carlo benchmark can be used for comparison.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
from math import exp, log, sqrt
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class MLMCConfig:
    s0: float = 100.0
    strike: float = 100.0
    maturity: float = 1.0
    risk_free_rate: float = 0.04
    dividend_yield: float = 0.00
    volatility: float = 0.20
    base_steps: int = 4
    max_level: int = 6
    seed: int = 42


config = MLMCConfig()
config

## 5. GBM path simulation at one level

For a given number of steps $M$, the exact GBM transition is:

$$
\begin{aligned}
S_{t+\Delta t} &= S_t \exp \Bigg[ (r-q-\frac12\sigma^2)\Delta t \\
&\quad + \sigma\sqrt{\Delta t}Z \Bigg]
\end{aligned}
$$
where:

$$
Z\sim\mathcal N(0,1)
$$

In [ ]:
def simulate_gbm_paths_from_normals(config: MLMCConfig, normals: np.ndarray) -> np.ndarray:
    """
    Simulate GBM paths from a supplied normal shock matrix.

    normals shape: (n_paths, n_steps)
    """
    n_paths, n_steps = normals.shape
    dt = config.maturity / n_steps

    log_increments = (
        (config.risk_free_rate - config.dividend_yield - 0.5 * config.volatility**2) * dt
        + config.volatility * np.sqrt(dt) * normals
    )

    log_paths = np.zeros((n_paths, n_steps + 1))
    log_paths[:, 0] = np.log(config.s0)
    log_paths[:, 1:] = np.log(config.s0) + np.cumsum(log_increments, axis=1)

    return np.exp(log_paths)


def arithmetic_asian_discounted_payoff(paths: np.ndarray, config: MLMCConfig) -> np.ndarray:
    """
    Discounted arithmetic Asian call payoff.

    The initial point is excluded from the arithmetic average.
    """
    average_price = paths[:, 1:].mean(axis=1)
    payoff = np.maximum(average_price - config.strike, 0.0)
    return np.exp(-config.risk_free_rate * config.maturity) * payoff

In [ ]:
rng = np.random.default_rng(config.seed)
test_normals = rng.standard_normal((5, 16))
test_paths = simulate_gbm_paths_from_normals(config, test_normals)
arithmetic_asian_discounted_payoff(test_paths, config)

## 6. Standard Monte Carlo benchmark

We first price the Asian option using standard Monte Carlo at a fine level.

This is the baseline that MLMC tries to beat.

In [ ]:
def standard_mc_asian_price(config: MLMCConfig, n_paths: int, level: int, seed: int = 123) -> dict:
    """
    Standard Monte Carlo Asian price at a fixed discretisation level.
    """
    rng = np.random.default_rng(seed)
    n_steps = config.base_steps * (2 ** level)

    normals = rng.standard_normal((n_paths, n_steps))
    paths = simulate_gbm_paths_from_normals(config, normals)
    payoff = arithmetic_asian_discounted_payoff(paths, config)

    price = float(np.mean(payoff))
    se = float(np.std(payoff, ddof=1) / np.sqrt(n_paths))

    return {
        "method": "standard_mc",
        "level": level,
        "n_steps": n_steps,
        "n_paths": n_paths,
        "price": price,
        "standard_error": se,
        "lower_95": price - 1.96 * se,
        "upper_95": price + 1.96 * se,
        "cost_proxy_path_steps": n_paths * n_steps
    }


standard_benchmark = standard_mc_asian_price(
    config=config,
    n_paths=200_000,
    level=config.max_level,
    seed=2026
)

standard_benchmark

## 7. Coupled coarse and fine paths

For MLMC, we need to simulate:

$$
(P_\ell, P_{\ell-1})
$$
on the same Brownian path.

If the fine level has twice as many steps as the coarse level, two fine Brownian increments combine into one coarse increment.

If fine increments are:

$$
\Delta W_1,\Delta W_2
$$
then the corresponding coarse increment is:

$$
\Delta W_c = \Delta W_1+\Delta W_2
$$
For normal shocks, this means:

$$
Z_c=\frac{Z_1+Z_2}{\sqrt{2}}
$$
This coupling is the heart of MLMC.

In [ ]:
def coupled_fine_coarse_payoffs(config: MLMCConfig, level: int, n_paths: int, seed: int) -> pd.DataFrame:
    """
    Simulate coupled fine and coarse Asian payoffs for an MLMC level.

    For level 0, only fine/coarse distinction is not used; returns P0.
    For level >= 1, returns P_l and P_{l-1} with Brownian coupling.
    """
    rng = np.random.default_rng(seed)

    fine_steps = config.base_steps * (2 ** level)

    if level == 0:
        normals = rng.standard_normal((n_paths, fine_steps))
        fine_paths = simulate_gbm_paths_from_normals(config, normals)
        fine_payoff = arithmetic_asian_discounted_payoff(fine_paths, config)

        return pd.DataFrame({
            "level": level,
            "fine_payoff": fine_payoff,
            "coarse_payoff": np.zeros_like(fine_payoff),
            "difference": fine_payoff,
            "fine_steps": fine_steps,
            "coarse_steps": 0
        })

    coarse_steps = fine_steps // 2

    fine_normals = rng.standard_normal((n_paths, fine_steps))
    coarse_normals = (
        fine_normals.reshape(n_paths, coarse_steps, 2).sum(axis=2) / np.sqrt(2.0)
    )

    fine_paths = simulate_gbm_paths_from_normals(config, fine_normals)
    coarse_paths = simulate_gbm_paths_from_normals(config, coarse_normals)

    fine_payoff = arithmetic_asian_discounted_payoff(fine_paths, config)
    coarse_payoff = arithmetic_asian_discounted_payoff(coarse_paths, config)

    return pd.DataFrame({
        "level": level,
        "fine_payoff": fine_payoff,
        "coarse_payoff": coarse_payoff,
        "difference": fine_payoff - coarse_payoff,
        "fine_steps": fine_steps,
        "coarse_steps": coarse_steps
    })


coupling_demo = coupled_fine_coarse_payoffs(config, level=3, n_paths=10_000, seed=11)

coupling_demo[["fine_payoff", "coarse_payoff", "difference"]].describe()

## 8. Level statistics

For each MLMC level, estimate:

$$
\mathbb E[Y_\ell]
$$
where:

$$
Y_0=P_0
$$
and:

$$
Y_\ell=P_\ell-P_{\ell-1}, \quad \ell\geq1
$$
We also estimate:

$$
\operatorname{Var}(Y_\ell)
$$
and a cost proxy:

$$
C_\ell \approx M_\ell+M_{\ell-1}
$$
The cost proxy is the number of simulated time steps per coupled sample.

In [ ]:
def estimate_level_statistics(config: MLMCConfig, pilot_paths: int = 20_000) -> pd.DataFrame:
    """
    Estimate mean, variance, and cost of MLMC level differences.
    """
    rows = []

    for level in range(config.max_level + 1):
        sample = coupled_fine_coarse_payoffs(
            config=config,
            level=level,
            n_paths=pilot_paths,
            seed=config.seed + 1000 + level
        )

        diff = sample["difference"].to_numpy()
        fine_steps = int(sample["fine_steps"].iloc[0])
        coarse_steps = int(sample["coarse_steps"].iloc[0])
        cost_per_sample = fine_steps + coarse_steps

        rows.append({
            "level": level,
            "pilot_paths": pilot_paths,
            "fine_steps": fine_steps,
            "coarse_steps": coarse_steps,
            "cost_per_sample": cost_per_sample,
            "mean_difference": float(np.mean(diff)),
            "variance_difference": float(np.var(diff, ddof=1)),
            "std_difference": float(np.std(diff, ddof=1)),
            "standard_error": float(np.std(diff, ddof=1) / np.sqrt(pilot_paths)),
            "mean_abs_difference": float(np.mean(np.abs(diff)))
        })

    return pd.DataFrame(rows)


level_stats = estimate_level_statistics(config, pilot_paths=20_000)

level_stats

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(level_stats["level"], np.abs(level_stats["mean_difference"]), marker="o", label="|mean level correction|")
plt.plot(level_stats["level"], level_stats["variance_difference"], marker="x", label="variance of correction")
plt.yscale("log")
plt.title("MLMC Level Difference Decay")
plt.xlabel("Level")
plt.ylabel("Log scale")
plt.legend()
plt.show()

## 9. MLMC telescoping estimator

The MLMC estimator is:

$$
\begin{aligned}
\hat P_{MLMC} &= \sum_{\ell=0}^L \frac{1}{N_\ell} \sum_{i=1}^{N_\ell}Y_\ell^{(i)}
\end{aligned}
$$
where:

$$
Y_0=P_0
$$
and:

$$
Y_\ell=P_\ell-P_{\ell-1}
$$
The variance is:

$$
\begin{aligned}
\operatorname{Var}(\hat P_{MLMC}) &= \sum_{\ell=0}^L \frac{V_\ell}{N_\ell}
\end{aligned}
$$
where:

$$
V_\ell=\operatorname{Var}(Y_\ell)
$$
The cost is:

$$
\sum_{\ell=0}^L N_\ell C_\ell
$$

## 10. Optimal sample allocation

For a target variance $\varepsilon^2$, the approximate optimal allocation is:

$$
N_\ell
\propto
\sqrt{\frac{V_\ell}{C_\ell}}
$$
More precisely:

$$
\begin{aligned}
N_\ell &= \left\lceil \frac{1}{\varepsilon^2} \sqrt{\frac{V_\ell}{C_\ell}} \sum_{j=0}^L \sqrt{V_jC_j} \right\rceil
\end{aligned}
$$
This allocates many samples to cheap/high-variance levels and fewer samples to expensive/low-variance levels.

In [ ]:
def optimal_mlmc_allocation(level_stats: pd.DataFrame, target_se: float, min_paths: int = 1_000) -> pd.DataFrame:
    """
    Compute approximate optimal sample counts per level for target standard error.
    """
    stats = level_stats.copy()

    V = stats["variance_difference"].clip(lower=1e-18).to_numpy()
    C = stats["cost_per_sample"].to_numpy()

    target_var = target_se ** 2
    allocation_factor = np.sum(np.sqrt(V * C)) / target_var

    N = np.ceil(allocation_factor * np.sqrt(V / C)).astype(int)
    N = np.maximum(N, min_paths)

    stats["allocated_paths"] = N
    stats["allocated_cost"] = stats["allocated_paths"] * stats["cost_per_sample"]
    stats["allocated_variance_contribution"] = stats["variance_difference"] / stats["allocated_paths"]

    return stats


allocation = optimal_mlmc_allocation(level_stats, target_se=0.01, min_paths=2_000)

allocation

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(allocation["level"].astype(str), allocation["allocated_paths"])
plt.title("Optimal MLMC Sample Allocation by Level")
plt.xlabel("Level")
plt.ylabel("Allocated paths")
plt.show()

plt.figure(figsize=(10, 6))
plt.bar(allocation["level"].astype(str), allocation["allocated_cost"])
plt.title("Allocated Cost Proxy by Level")
plt.xlabel("Level")
plt.ylabel("Path-step cost proxy")
plt.show()

## 11. Run the MLMC estimator

We now run the MLMC estimator using the allocated paths.

For each level:

1. simulate $N_\ell$ coupled samples;
2. estimate $\mathbb E[Y_\ell]$;
3. sum the level means.

In [ ]:
def run_mlmc_estimator(config: MLMCConfig, allocation: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    """
    Run MLMC estimator using specified allocation.
    """
    rows = []

    for row in allocation.itertuples():
        level = int(row.level)
        n_paths = int(row.allocated_paths)

        sample = coupled_fine_coarse_payoffs(
            config=config,
            level=level,
            n_paths=n_paths,
            seed=config.seed + 5000 + level
        )

        diff = sample["difference"].to_numpy()
        cost_per_sample = int(sample["fine_steps"].iloc[0] + sample["coarse_steps"].iloc[0])

        rows.append({
            "level": level,
            "n_paths": n_paths,
            "fine_steps": int(sample["fine_steps"].iloc[0]),
            "coarse_steps": int(sample["coarse_steps"].iloc[0]),
            "cost_per_sample": cost_per_sample,
            "level_mean": float(np.mean(diff)),
            "level_variance": float(np.var(diff, ddof=1)),
            "level_standard_error": float(np.std(diff, ddof=1) / np.sqrt(n_paths)),
            "variance_contribution": float(np.var(diff, ddof=1) / n_paths),
            "cost_proxy": int(n_paths * cost_per_sample)
        })

    level_results = pd.DataFrame(rows).sort_values("level")

    price = float(level_results["level_mean"].sum())
    variance_estimate = float(level_results["variance_contribution"].sum())
    se = sqrt(variance_estimate)

    result = {
        "method": "mlmc",
        "max_level": int(level_results["level"].max()),
        "price": price,
        "standard_error": se,
        "lower_95": price - 1.96 * se,
        "upper_95": price + 1.96 * se,
        "total_paths": int(level_results["n_paths"].sum()),
        "cost_proxy_path_steps": int(level_results["cost_proxy"].sum())
    }

    return result, level_results


mlmc_result, mlmc_level_results = run_mlmc_estimator(config, allocation)

mlmc_result

In [ ]:
mlmc_level_results

## 12. Compare MLMC with standard Monte Carlo

We compare MLMC with standard Monte Carlo at the finest level.

The standard MC run is sized so that its cost proxy is similar to the MLMC cost proxy.

This makes the comparison fairer than comparing path counts alone.

In [ ]:
def standard_mc_with_cost_budget(config: MLMCConfig, level: int, cost_budget: int, seed: int = 777) -> dict:
    """
    Run standard MC at a fixed level using approximately the same cost budget.
    """
    n_steps = config.base_steps * (2 ** level)
    n_paths = max(1_000, int(cost_budget // n_steps))

    return standard_mc_asian_price(
        config=config,
        n_paths=n_paths,
        level=level,
        seed=seed
    )


standard_same_cost = standard_mc_with_cost_budget(
    config=config,
    level=config.max_level,
    cost_budget=mlmc_result["cost_proxy_path_steps"],
    seed=888
)

comparison = pd.DataFrame([
    mlmc_result,
    standard_same_cost,
    standard_benchmark
])

comparison

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(comparison["method"] + "_L" + comparison["max_level"].fillna(comparison["level"]).astype(str), comparison["standard_error"])
plt.title("Standard Error Comparison")
plt.xlabel("Method")
plt.ylabel("Estimated standard error")
plt.xticks(rotation=20, ha="right")
plt.show()

plt.figure(figsize=(10, 6))
plt.bar(comparison["method"] + "_L" + comparison["max_level"].fillna(comparison["level"]).astype(str), comparison["cost_proxy_path_steps"])
plt.title("Cost Proxy Comparison")
plt.xlabel("Method")
plt.ylabel("Path-step cost proxy")
plt.xticks(rotation=20, ha="right")
plt.show()

## 13. Bias diagnostic

MLMC estimates:

$$
\mathbb E[P_L]
$$
not the exact continuous-time payoff.

There is still discretisation bias:

$$
\mathbb E[P_L]-\mathbb E[P]
$$
A practical diagnostic is the size of the last level correction:

$$
|\mathbb E[P_L-P_{L-1}]|
$$
If the last correction is small compared with the target error, the level is probably fine enough.

This is not a proof, but it is a useful engineering rule.

In [ ]:
bias_diagnostic = pd.Series({
    "last_level": int(level_stats["level"].max()),
    "estimated_last_level_mean_correction": float(level_stats.iloc[-1]["mean_difference"]),
    "absolute_last_level_correction": float(abs(level_stats.iloc[-1]["mean_difference"])),
    "mlmc_standard_error": mlmc_result["standard_error"],
    "last_correction_over_standard_error": float(abs(level_stats.iloc[-1]["mean_difference"]) / mlmc_result["standard_error"])
})

bias_diagnostic

## 14. Cost and variance anatomy

MLMC works when high-level corrections have low variance.

A useful diagnostic table compares:

- variance by level;
- cost by level;
- allocated samples;
- variance contribution after allocation.

In [ ]:
anatomy = mlmc_level_results.merge(
    level_stats[["level", "mean_abs_difference", "variance_difference"]],
    on="level",
    how="left"
)

anatomy[[
    "level",
    "fine_steps",
    "n_paths",
    "level_mean",
    "variance_difference",
    "level_variance",
    "variance_contribution",
    "cost_proxy",
    "mean_abs_difference"
]]

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(anatomy["level"], anatomy["variance_contribution"], marker="o")
plt.yscale("log")
plt.title("MLMC Variance Contribution by Level After Allocation")
plt.xlabel("Level")
plt.ylabel("Variance contribution")
plt.show()

## 15. MLMC versus grid refinement

We can run MLMC at increasing maximum levels and inspect how the estimate changes.

This shows how the discretisation level affects the final result.

In [ ]:
def mlmc_by_max_level_experiment(config: MLMCConfig, max_levels, pilot_paths=8_000, target_se=0.02):
    rows = []

    for L in max_levels:
        cfg_L = MLMCConfig(**{**asdict(config), "max_level": int(L)})
        stats_L = estimate_level_statistics(cfg_L, pilot_paths=pilot_paths)
        alloc_L = optimal_mlmc_allocation(stats_L, target_se=target_se, min_paths=1_000)
        result_L, _ = run_mlmc_estimator(cfg_L, alloc_L)

        rows.append({
            "max_level": int(L),
            "price": result_L["price"],
            "standard_error": result_L["standard_error"],
            "cost_proxy_path_steps": result_L["cost_proxy_path_steps"],
            "last_level_mean_correction": float(stats_L.iloc[-1]["mean_difference"]),
            "abs_last_level_mean_correction": float(abs(stats_L.iloc[-1]["mean_difference"]))
        })

    return pd.DataFrame(rows)


mlmc_level_experiment = mlmc_by_max_level_experiment(
    config=config,
    max_levels=[2, 3, 4, 5, 6],
    pilot_paths=8_000,
    target_se=0.02
)

mlmc_level_experiment

In [ ]:
plt.figure(figsize=(10, 6))
plt.errorbar(
    mlmc_level_experiment["max_level"],
    mlmc_level_experiment["price"],
    yerr=1.96 * mlmc_level_experiment["standard_error"],
    marker="o",
    capsize=4
)
plt.title("MLMC Estimate by Maximum Level")
plt.xlabel("Maximum level")
plt.ylabel("Asian option price")
plt.show()

## 16. Practical checklist for MLMC

Before using MLMC in production, check:

1. **Coupling**  
   Are coarse and fine paths driven by the same Brownian path?

2. **Telescoping identity**  
   Is the estimator actually estimating:

$$
\mathbb E[P_0]+\sum_{\ell=1}^L\mathbb E[P_\ell-P_{\ell-1}]?
$$
3. **Variance decay**  
   Does $\operatorname{Var}(P_\ell-P_{\ell-1})$ decrease with $\ell$?

4. **Bias diagnostic**  
   Is the final level correction small enough?

5. **Cost model**  
   Is cost measured realistically?

6. **Sample allocation**  
   Are more samples allocated to cheap/high-variance levels?

7. **Payoff discontinuity**  
   Does the payoff have barriers or discontinuities that weaken coupling?

8. **Confidence interval**  
   Does the output report standard error?

9. **Reproducibility**  
   Are random seeds and level allocations saved?

10. **Benchmarking**  
   Is MLMC compared against standard MC at similar cost?

## 17. Limitations

### 17.1 Asian option only

This notebook uses an arithmetic Asian option because it is path-dependent and smooth enough for clean MLMC demonstration.

Barrier options require more care because discontinuous payoff indicators can weaken variance decay.

### 17.2 GBM only

The simulation uses GBM.

MLMC can be extended to stochastic volatility, local volatility, and jump models, but coupling becomes more complex.

### 17.3 Simple allocation

The sample allocation uses pilot estimates.

Production systems may update allocation adaptively.

### 17.4 Cost proxy is simplified

We use path-step count as a cost proxy.

Real runtime depends on vectorisation, memory bandwidth, random number generation, hardware, and payoff logic.

### 17.5 Bias is estimated heuristically

The final level correction is a practical bias diagnostic, not a mathematical proof.

### 17.6 No quasi-Monte Carlo

The notebook does not combine MLMC with quasi-Monte Carlo, though the combination can be powerful.

## 18. Research frontier and current directions

Important extensions include:

1. **Adaptive MLMC**  
   Automatically choose levels and samples to meet a target RMSE.

2. **MLMC for SDE discretisation schemes**  
   Compare Euler, Milstein, and exact schemes.

3. **MLMC Greeks**  
   Estimate Greeks efficiently using pathwise or likelihood-ratio methods.

4. **MLMC for barrier options**  
   Use Brownian bridge corrections to improve coupling.

5. **MLMC under Heston**  
   Couple stochastic variance paths across levels.

6. **MLMC with jumps**  
   Couple jump times and jump sizes across levels.

7. **Quasi-Monte Carlo MLMC**  
   Combine low-discrepancy sequences with multilevel estimators.

8. **GPU MLMC**  
   Parallelise level simulations and allocation loops.

9. **Nested MLMC**  
   Useful for XVA, portfolio risk, and exposure simulation.

10. **Unbiased MLMC**  
   Randomise levels to remove discretisation bias.

## 19. Suggested follow-up notebooks

This notebook naturally leads to:

1. **02_14_local_volatility_dupire_surface**  
   More advanced model construction from volatility surfaces.

2. **02_15_jump_diffusion_pide_pricer**  
   Comparing PDE/PIDE methods with Monte Carlo.

3. **02_08_dynamic_delta_hedging_simulation**  
   Hedging simulations where path resolution matters.

4. **04_06_var_cvar_and_stress_testing**  
   MLMC-style simulation efficiency for portfolio risk.

5. **06_10_numba_cpp_pde_kernels**  
   Acceleration of numerical pricing kernels.

6. **06_12_gpu_monte_carlo_engine**  
   GPU Monte Carlo and MLMC benchmarking.

7. **07_02_volatility_surface_pricing_and_alpha**  
   Capstone pricing engine comparison across surface models.

## 20. Saving outputs

The notebook saves:

1. standard MC benchmark;
2. MLMC level statistics;
3. optimal allocation;
4. MLMC level results;
5. MLMC vs standard MC comparison;
6. bias diagnostics;
7. max-level experiment;
8. manifest.

In [ ]:
output_dir = Path("data/processed/multilevel_monte_carlo_pricing_v1")
output_dir.mkdir(parents=True, exist_ok=True)

standard_benchmark_path = output_dir / "standard_mc_benchmark.csv"
level_stats_path = output_dir / "mlmc_level_statistics.csv"
allocation_path = output_dir / "mlmc_optimal_allocation.csv"
mlmc_level_results_path = output_dir / "mlmc_level_results.csv"
comparison_path = output_dir / "mlmc_vs_standard_mc_comparison.csv"
bias_path = output_dir / "bias_diagnostic.csv"
max_level_experiment_path = output_dir / "mlmc_max_level_experiment.csv"
manifest_path = output_dir / "manifest.json"

pd.DataFrame([standard_benchmark]).to_csv(standard_benchmark_path, index=False)
level_stats.to_csv(level_stats_path, index=False)
allocation.to_csv(allocation_path, index=False)
mlmc_level_results.to_csv(mlmc_level_results_path, index=False)
comparison.to_csv(comparison_path, index=False)
bias_diagnostic.to_frame("value").to_csv(bias_path)
mlmc_level_experiment.to_csv(max_level_experiment_path, index=False)

manifest = {
    "dataset_name": "multilevel_monte_carlo_pricing_outputs",
    "schema_version": "multilevel_monte_carlo_pricing_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "standard_benchmark": standard_benchmark,
    "mlmc_result": mlmc_result,
    "bias_diagnostic": bias_diagnostic.to_dict(),
    "notes": [
        "Payoff is a discounted arithmetic Asian call under GBM.",
        "Fine and coarse paths are coupled by aggregating Brownian increments.",
        "MLMC allocation uses pilot variance and cost estimates.",
        "Cost proxy is number of path time steps.",
        "Final level correction is used as a practical bias diagnostic."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

standard_benchmark_path, level_stats_path, allocation_path, comparison_path, manifest_path

## 21. Summary

This notebook implemented Multilevel Monte Carlo for an Asian option.

It showed how to:

1. simulate GBM paths;
2. price an arithmetic Asian option with standard Monte Carlo;
3. construct coupled fine/coarse paths;
4. estimate MLMC level differences;
5. measure variance decay and cost per level;
6. compute optimal sample allocation;
7. build the MLMC telescoping estimator;
8. compare MLMC against standard MC at similar cost;
9. diagnose discretisation bias using the final level correction;
10. save MLMC diagnostics and outputs.

The key computational takeaway:

> MLMC reduces cost by spending most samples on cheap levels and only a few samples on expensive high-resolution corrections.

The key financial takeaway:

> For path-dependent derivatives, numerical efficiency depends not only on the stochastic model, but on how cleverly the simulation estimator is constructed.

## 22. Further reading

- Giles, M. B. "Multilevel Monte Carlo Path Simulation."
- Giles, M. B. "Multilevel Monte Carlo Methods."
- Glasserman, *Monte Carlo Methods in Financial Engineering*.
- Jäckel, *Monte Carlo Methods in Finance*.
- Research on MLMC for Greeks, Heston models, jump diffusions, barrier options, and XVA.